# 08. Документация: отчёт о модели, model card, описание генерации данных, итоговый отчёт

Все числа подставляются автоматически из артефактов предыдущих ноутбуков (`docs/ml/metrics.json`, манифест датасета, `training_meta.json`) — отчёт не расходится с кодом.
Выход: `docs/ml/model-report.md`, `docs/ml/model-card.md`, `docs/ml/data-generation.md`, `ОТЧЁТ_DS.md` (в корне проекта), `Makefile`, `risk-scoring/requirements-ds.lock.txt`.

In [1]:
import sys, json, subprocess, platform, importlib.metadata as im
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "risk-scoring")); sys.path.insert(0, str(ROOT))
from riskscore.features import FEATURE_META, FEATURE_COLUMNS
m = json.load(open(ROOT / "docs/ml/metrics.json", encoding="utf-8"))
man = json.load(open(ROOT / "data/generated/manifest.json", encoding="utf-8"))
tm = json.load(open(ROOT / "risk-scoring/models/training_meta.json", encoding="utf-8"))
card = json.load(open(ROOT / "risk-scoring/models/scorecard_v1.json", encoding="utf-8"))
scale = json.load(open(ROOT / "risk-scoring/models/rating_scale_v1.json", encoding="utf-8"))["grades"]
demo = pd.read_csv(ROOT / "data/fixtures/demo_cases.csv", dtype={"inn": str})
LR = "WOE-LR + скор-карта (основная)"
T = m["metrics"][f"{LR}|test"]; V = m["metrics"][f"{LR}|valid"]; TR = m["metrics"][f"{LR}|train"]
ci = m["test_ci"]
# тесты
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no", str(ROOT / "risk-scoring/tests")], capture_output=True, text=True, cwd=ROOT)
n_tests = int([l for l in r.stdout.splitlines() if "passed" in l][-1].split()[0]); tests_line = [l for l in r.stdout.splitlines() if "passed" in l][-1]
print("pytest:", tests_line)
# статистика датасета для текста (считается, а не вписывается руками)
from sklearn.metrics import roc_auc_score
ds = pd.read_csv(ROOT / "data/generated/applications_raw.csv", dtype={"inn": str, "ogrn": str})
oracle_gini = 2 * roc_auc_score(ds.default_flag, ds.oracle_pd) - 1
o90 = ds.groupby(ds.bki_overdue_90_cnt.clip(upper=2)).default_flag.mean()
rev_ltm = sum(ds[f"revenue_q{k}"] for k in range(5, 9)); to12 = sum(ds[f"turnover_credit_m{k}"] for k in range(1, 13))
corr_turn = float(np.corrcoef(np.log(rev_ltm), np.log(to12))[0, 1]); no_hist = float((ds.bki_has_history == 0).mean())
print(f"оракул Gini {oracle_gini:.3f} | просрочки 90+: {o90.round(3).to_dict()} | corr {corr_turn:.3f} | нет КИ {no_hist:.1%}")
lab = lambda cs: ", ".join(FEATURE_META[c]["label"].lower() for c in cs)
top_perm = lab(list(m["permutation_top"])[:6])
top_card = lab([k for _, k in sorted(((max(v["points"]) - min(v["points"]), k) for k, v in card["features"].items()), reverse=True)[:6]])
_rt = pd.DataFrame(m["rating_test"]); n_in_ci = int(((_rt.mean_pd >= _rt.ci_lo) & (_rt.mean_pd <= _rt.ci_hi)).sum())
in_ci_txt = "во всех градациях" if n_in_ci == len(_rt) else f"в {n_in_ci} из {len(_rt)} градаций"
assert tm["calibration"]["woe_logit"] == "platt"
md = lambda df: "\n".join(["| " + " | ".join(map(str, df.columns)) + " |", "|" + "---|" * len(df.columns)] + ["| " + " | ".join(map(str, r_)) + " |" for r_ in df.to_numpy()])
n_cards = len(card["features"]); gap = m["gini_gap_train_test"]
print("модель:", m["model"], "| Gini test:", round(T["gini"], 3), "| тестов:", n_tests)

pytest: 27 passed, 2 warnings in 9.49s


оракул Gini 0.655 | просрочки 90+: {0.0: 0.095, 1.0: 0.3, 2.0: 0.25} | corr 0.979 | нет КИ 17.2%
модель: v1.0.0-woe-logit | Gini test: 0.659 | тестов: 27


## Версии библиотек (lock) и Makefile

In [2]:
pk = ["numpy", "pandas", "scikit-learn", "scipy", "matplotlib", "joblib", "pytest", "nbformat", "nbclient", "ipykernel"]
vers = {p: im.version(p) for p in pk}
lock = ["# Зафиксированные версии (Python " + platform.python_version() + "); сгенерировано ноутбуком 08", *[f"{p}=={v}" for p, v in vers.items()]]
(ROOT / "risk-scoring/requirements-ds.lock.txt").write_text("\n".join(lock) + "\n", encoding="utf-8")
(ROOT / "Makefile").write_text("""# Воспроизведение DS-части (все данные синтетические). Запуск: make data | train | report | test | all
PY ?= python3
NB = $(PY) -m jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=900

data:    ## генератор синтетики, побайтово воспроизводим
\t$(PY) -m data.generate --seed 42

train:   ## признаки -> модели -> скор-карта и рейтинги (артефакты в risk-scoring/models)
\t$(NB) notebooks/01_data_generation.ipynb
\t$(NB) notebooks/02_features.ipynb
\t$(NB) notebooks/03_modeling.ipynb
\t$(NB) notebooks/04_scorecard_rating.ipynb

report:  ## метрики, графики, фикстуры, документация
\t$(NB) notebooks/05_validation.ipynb
\t$(NB) notebooks/06_scoring_engine.ipynb
\t$(NB) notebooks/07_demo_cases_fixtures.ipynb
\t$(NB) notebooks/08_documentation.ipynb

test:
\t$(PY) -m pytest -q risk-scoring/tests

all: train report test
.PHONY: data train report test all
""", encoding="utf-8")
print(open(ROOT / "risk-scoring/requirements-ds.lock.txt").read())

# Зафиксированные версии (Python 3.12.10); сгенерировано ноутбуком 08
numpy==2.3.0
pandas==2.3.0
scikit-learn==1.7.0
scipy==1.15.3
matplotlib==3.10.3
joblib==1.5.1
pytest==9.1.1
nbformat==5.10.4
nbclient==0.10.2
ipykernel==6.29.5



## `docs/ml/data-generation.md`

In [3]:
DG = f"""# Генерация синтетических данных

> **Все данные синтетические**: сгенерированы командой хакатона скриптом `data/generate.py`; к реальным лицам и организациям не относятся. Реальные ИНН, названия, ФИО и выписки не использовались. По условиям ТЗ покупные/закрытые датасеты запрещены.

## Воспроизводимость
`python -m data.generate --seed 42` создаёт `data/generated/applications_raw.csv` ({man['train_rows']} заявок для обучения) и `demo_pool_raw.csv` ({man['demo_pool_rows']} компаний-кандидатов для демо) + `manifest.json` с SHA-256. Повторный запуск даёт **побайтово одинаковые** файлы (проверено в ноутбуке 01). Версия датасета: `{man['dataset_version']}`, seed {man['seed']}.

## Идентификаторы
ИНН (10 цифр для ООО, 12 для ИП) и ОГРН/ОГРНИП считаются по официальным алгоритмам контрольных сумм, поэтому проходят валидатор. Гарантия несуществования: код региона = `00` (в ИНН — позиции 1–2, в ОГРН — позиции 4–5); реальных организаций с таким кодом нет. Ведущие нули важны — ИНН хранится и читается **только строкой**.

## Структура (1 строка = 1 компания-заявка)
| Блок | Что содержит |
|---|---|
| Профиль | форма (ООО/ИП), дата регистрации, ОКВЭД (11 отраслей), регион (17), численность, налоговый режим, статус (в т.ч. 1% «в ликвидации») |
| Финансы, 8 кварталов | выручка, себестоимость, EBITDA, чистая прибыль, активы, обязательства, капитал, краткосрочные обязательства, оборотные активы, дебиторка, кредиторка |
| Обороты, 12 месяцев | поступления, списания, остаток на конец месяца; число контрагентов; срок обслуживания в банке |
| БКИ | открытые/закрытые кредиты, суммы, текущий долг, платёж, просрочки 30/60/90+, макс. просрочка, текущая просрочка, дефолты, запросы за 3/12 мес. Нет истории (~{no_hist:.0%}) → поля `NaN`, а не 0 |
| ФНС | налоговый долг, блокировка счетов, массовый адрес/руководитель, дисквалификация, банкротство |
| Заявка | продукт (4), сумма, срок, индикативная ставка, цель, собственный взнос, обеспечение |
| Цель | `default_flag` — дефолт (90+ дней просрочки) в течение 12 месяцев после заявки |

## Как устроена генерация
1. **Скрытое «качество бизнеса» q ~ N(0,1)** у каждой компании. Оно влияет на маржу, рост, волатильность, структуру капитала и платёжную дисциплину, но *не наблюдается* — модель видит только проявления.
2. **Финансы с настоящей арифметикой**: `EBITDA = валовая прибыль − постоянные расходы` (операционный рычаг), `чистая прибыль = EBITDA − амортизация − проценты − налог по режиму (УСН 6% / УСН 15% / ОСН 20% / ПСН)`, `капитал_t = капитал_(t−1) + прибыль − дивиденды`, **`активы = обязательства + капитал` выполняется точно** (макс. невязка 1 ₽ — округление). Сезонность и «шоки» выручки (падение на 15–50% в последних кварталах) зависят от q.
3. **Обороты по счёту** строятся из выручки (корреляция логарифмов LTM-выручки и оборотов {corr_turn:.2f}), БКИ — из долговой нагрузки по балансу и q.
4. **Порождающая модель риска** (`default_flag`): логит риска = `SIGNAL · (наблюдаемые факторы) + дрейф по году − 0.35·q + ε`, ε ~ N(0, 1.5²), свободный член подбирается бисекцией под среднюю долю дефолтов ≈ 10%; исход — Bernoulli(PD).

   Наблюдаемые факторы (знак — влияние на риск): совокупная долговая нагрузка (платежи/EBITDA) `+1.1·min(·,2.5)`; просрочки 90+ `+0.9`, 60+ `+0.35` за каждую (до 3), 30+ `+0.2` (до 4); текущая просрочка `+0.8`; блокировка счёта `+1.5`; банкротство `+1.8`; дисквалификация `+1.2`; массовый адрес `+0.6`, руководитель `+0.5`; налоговый долг к месячной выручке `+1.5·min(·,1)`; волатильность выручки `+2.0·(CV−0.12)`; рост выручки `−1.2·рост`; возраст `−0.45·(ln мес − ln 36)`; отрицательный капитал `+0.8`, автономия `−1.0·коэф.`; ликвидность `−0.4·(CR−1)`; запросы в БКИ `+0.18` за запрос; отрасль (−0.35…+0.30); нет КИ `+0.25`; ≥3 закрытых кредитов без просрочек `−0.3`; обеспечение `−0.9·покрытие`; убыточные кварталы `+0.25` за квартал; рентабельность по EBITDA `−3·(маржа−10%)`. Множитель `SIGNAL = 0.55` подобран так, чтобы Gini «оракула» (знающего точную формулу наблюдаемой части) был ≈ {oracle_gini:.2f} — реалистичный для МСБ, а не идеальный.
5. **Шум и несовершенство**: скрытые q и ε делают прогноз принципиально неточным — модель не может достичь Gini = 1.

## Проверки качества синтетики (ноутбук 01, `risk-scoring/tests`)
- доля дефолтов **{man['train_default_rate']:.1%}** (требование 6–15%);
- ИНН/ОГРН: 100% проходят контрольные суммы, 100% в диапазоне региона `00`, все уникальны;
- баланс сходится, активы > 0, прибыль ≤ выручки, себестоимость ≤ выручки;
- монотонность: доля дефолтов растёт с нагрузкой, просрочками 90+ ({o90[0]:.1%} → {o90[1]:.1%} → {o90[2]:.1%} при 0 / 1 / 2+ просрочках 90+), волатильностью, налоговым долгом, блокировкой; падает с возрастом бизнеса;
- ориентир качества: Gini оракула = {oracle_gini:.2f} (по наблюдаемой части; реальная модель может превысить его за счёт прокси скрытого качества).

## Ограничения
Закономерности заданы самими авторами (порождающая модель), поэтому модель, обученная на этих данных, *восстанавливает наши же допущения*. Это демонстрирует методологию, а **метрики нельзя переносить на реальный портфель**. Рыночные параметры (ставки, маржи отраслей, сезонность) — экспертные допущения; открытыми справочниками использованы только классификатор ОКВЭД и названия регионов РФ.
"""
(ROOT / "docs/ml/data-generation.md").write_text(DG, encoding="utf-8")
print(len(DG.splitlines()), "строк")

39 строк


## `docs/ml/model-report.md`

In [4]:
def fmt_metrics(name, mm, k): return f"| {name} | {k} | {mm['n']} | {m['default_rate'][k.strip('*')]:.2%} | {mm['auc']:.3f} | {mm['gini']:.3f} | {mm['ks']:.3f} | {mm['brier']:.4f} | {mm['ece']:.4f} |"
rt = pd.DataFrame(m["rating_test"])
rt_t = pd.DataFrame({"Рейтинг": rt.grade, "Заявок": rt.n, "Доля": (rt.share * 100).round(1).astype(str) + "%", "Дефолтов": rt.defaults, "Средний PD": (rt.mean_pd * 100).round(2).astype(str) + "%",
                     "Факт. доля дефолтов": (rt.observed * 100).round(2).astype(str) + "%", "95% ДИ факта": [f"{a*100:.1f}–{b*100:.1f}%" for a, b in zip(rt.ci_lo, rt.ci_hi)]})
seg = pd.DataFrame(m["segments"]); seg_t = pd.DataFrame({"Сегмент": seg.segment, "Значение": seg.value, "n": seg.n, "Доля дефолтов": (seg.default_rate * 100).round(1).astype(str) + "%",
                    "Средний PD": (seg.mean_pd * 100).round(1).astype(str) + "%", "Gini": seg.gini.map(lambda x: "—" if x is None or pd.isna(x) else f"{x:.2f}")})
gseg = seg.gini.dropna()
psi = m["psi"]; sens = pd.DataFrame({"Стресс-изменение": list(m["sensitivity"]), "Δ среднего PD": [f"{v:+.4f}" for v in m["sensitivity"].values()]})
perm = pd.DataFrame({"Признак": [FEATURE_META[c]["label"] for c in m["permutation_top"]], "Падение Gini при перемешивании": [f"{v:.4f}" for v in m["permutation_top"].values()]})
cal = pd.DataFrame(m["calibration_test"]); cal_t = pd.DataFrame({"Дециль": range(1, 11), "Средний прогноз PD": (cal.mean_pd * 100).round(2).astype(str) + "%", "Факт": (cal.observed * 100).round(2).astype(str) + "%"})
REPORT = f"""# Отчёт о модели оценки кредитного риска

> ⚠️ **Модель обучена на синтетических данных.** Метрики характеризуют качество методологии на искусственной выборке и **не переносятся на реальный портфель**. Это демонстрация подхода, а не боевая модель.

Модель: `{m['model']}` (основная, WOE-логистическая регрессия + калибровка Платта + скор-карта). Датасет `{m['dataset']}` (seed {m['seed']}). Воспроизведение: `make train && make report`; графики — `docs/ml/img/`.

## 1. Выборки и протокол
Одна компания = одна заявка, разбиение стратифицировано **по компаниям** (повторных наблюдений нет): train {m['sizes']['train']} / valid {m['sizes']['valid']} / test {m['sizes']['test']}; доля дефолтов {m['default_rate']['train']:.2%} / {m['default_rate']['valid']:.2%} / {m['default_rate']['test']:.2%}. Train — обучение и WOE-биннинг; valid — подбор гиперпараметров, калибровка; границы рейтингов подбирались на train+valid; **test использован один раз для итоговой оценки**. Утечки исключены: цель и производные (`oracle_pd`) не входят в признаки (проверено кодом); максимальный одномерный Gini отдельного признака ниже 0.7 (проверка в ноутбуке 02), «подозрительно сильных» признаков нет.

## 2. Различающая способность и точность (ТЗ 2.4.1–2.4.3)
| Модель | Выборка | n | Доля дефолтов | AUC | Gini | KS | Brier | ECE |
|---|---|---|---|---|---|---|---|---|
{fmt_metrics('WOE-LR (основная)', TR, 'train')}
{fmt_metrics('WOE-LR (основная)', V, 'valid')}
{fmt_metrics('**WOE-LR (основная)**', T, '**test**')}

**Основная модель на test (95% бутстрэп-ДИ):** Gini **{T['gini']:.3f}** [{ci['gini'][0]:.3f}; {ci['gini'][1]:.3f}], KS **{T['ks']:.3f}** [{ci['ks'][0]:.3f}; {ci['ks'][1]:.3f}], Brier **{T['brier']:.4f}** [{ci['brier'][0]:.4f}; {ci['brier'][1]:.4f}], ROC-AUC {T['auc']:.3f}. Требование ТЗ Gini ≥ 0.45 **выполнено** (нижняя граница ДИ {ci['gini'][0]:.3f} тоже выше порога). Brier Skill Score относительно «всем средний PD» = {m['brier_skill']:.3f}. Разрыв Gini train→test {gap:+.3f} ({'переобучения не видно' if gap < 0.03 else 'умеренное переобучение'}).

![ROC и KS](img/05_roc_ks.png)

## 3. Калибровка (ТЗ 2.3.4, 2.4.3)
Метод выбран кросс-валидацией внутри valid: Platt (параметры a={card['calibration_a']:.3f}, b={card['calibration_b']:.3f}); isotonic не дал выигрыша в Brier > 2%. Средний PD на test {T['mean_pd']:.2%} при факте {m['default_rate']['test']:.2%}; ECE {T['ece']:.4f}.

{md(cal_t)}

![Calibration curve](img/05_calibration.png)

## 4. Таблица рейтингов на отложенной выборке (ТЗ 2.4.5)
Фактическая доля дефолтов **монотонно растёт** от AAA к D: на test — {m['rating_monotone']['test']}, на всей выборке из 12 000 заявок — {m['rating_monotone']['all']}. Средний PD модели попадает в 95% доверительный интервал (Уилсон) факта {in_ci_txt} (в верхних рейтингах — единицы дефолтов, интервалы широки).

{md(rt_t)}

![Рейтинги](img/05_rating_table.png)

## 5. Стабильность (ТЗ 2.4.4)
- **PSI скора:** train→test **{psi['score_train_test']:.4f}**, train→valid {psi['score_train_valid']:.4f}; по годам заявок 2023/2024/2025: {psi['by_year']['2023']:.4f} / {psi['by_year']['2024']:.4f} / {psi['by_year']['2025']:.4f} (порог «стабильно» < 0.10).
- **PSI признаков** train→test: максимум {psi['features_max']:.4f}, признаков с PSI > 0.1: {psi['features_gt_0_1']}.
- **Сегменты (test, valid+test для годов).** Gini по сегментам (n ≥ 150 и ≥ 10 дефолтов): от {gseg.min():.2f} до {gseg.max():.2f}, сегментов с Gini ≥ 0.45: {int((gseg >= 0.45).sum())} из {len(gseg)}. Различия частично объясняются малым размером сегментов (шум).

{md(seg_t)}

## 6. Какие факторы влияют на PD и как это проверено
**Скор-карта** содержит {n_cards} признаков. Самые «тяжёлые» по размаху баллов: {top_card}. По падению Gini при перемешивании на test (permutation importance) лидируют: {top_perm}.

{md(perm)}

Для объяснения клиенту используются баллы скор-карты (вклад каждого признака), а не рейтинг важности.

![Важность признаков](img/05_importance.png)

**Проверка здравого смысла** (ухудшаем один фактор у реальных заявок test, смотрим средний PD) — все знаки верные, PD реагирует и на сумму, и на срок заявки:

{md(sens)}

Дополнительно автотесты (`pytest`, {n_tests} шт.) проверяют, что рост суммы заявки, текущих платежей, просрочек и блокировка счёта **не снижают PD**, а вывод детерминирован.

## 7. Ограничения и честные оговорки (ТЗ 2.4.8)
1. Данные синтетические, а закономерности заданы нашей порождающей моделью, — высокое качество частично «подсказано» структурой данных. Метрики **не переносятся** на реальность.
2. Редкие события (блокировка счёта, банкротство) представлены десятками наблюдений; их веса в модели оценены неточно.
3. Верхние рейтинги (AAA/AA) содержат единицы дефолтов — различие между соседними градациями статистически слабое (см. доверительные интервалы).
4. Скор отражает риск дефолта за 12 мес.; он не учитывает макроэкономику, залоги и поведение после выдачи.
"""
(ROOT / "docs/ml/model-report.md").write_text(REPORT, encoding="utf-8")
print(len(REPORT.splitlines()), "строк")

130 строк


## `docs/ml/model-card.md`

In [5]:
feat_groups = pd.Series({k: v["group"] for k, v in FEATURE_META.items()}).value_counts().sort_index()
CARD = f"""# Model card: оценка кредитного риска МСБ

> **Синтетика.** Модель обучена и проверена только на искусственных данных (`{m['dataset']}`); к реальным заёмщикам неприменима без переобучения и независимой валидации.

| | |
|---|---|
| Версия | `{m['model']}` |
| Дата обучения | {pd.Timestamp.today().strftime('%Y-%m-%d')} |
| Тип | бинарная классификация: вероятность дефолта (90+ дней) в течение 12 мес. после заявки |
| Назначение | оценка риска в кредитном конвейере малого/микробизнеса: `pd`, `riskScore`, `ratingGrade`, рекомендация лимита/ставки/срока, флаги политики, объяснение факторов |
| Владелец | роль «Classic ML Engineer» (скоринг и данные) |

## Что делает и чего НЕ делает
Модель даёт **оценку риска и рекомендацию**. Она **не принимает кредитное решение**: пороги одобрения, лимиты продуктов и hard-stop правила — конфигурация `credit-core`. Флаги политики (`policyFlags`) выставляются движком, решение по ним принимает Backend.

**Нельзя использовать:** для реальных кредитных решений; для физлиц; для сегментов, которых нет в данных (крупный бизнес, финансовые организации); как единственное основание для отказа без права на пересмотр человеком; для дискриминации (в признаках нет персональных характеристик).

## Данные
{man['train_rows']} синтетических заявок (60/20/20 train/valid/test по компаниям), доля дефолтов {man['train_default_rate']:.1%}, период заявок 2023-01…2025-09. Генерация и проверки — `docs/ml/data-generation.md`.

## Признаки
{len(FEATURE_COLUMNS)} признаков в 5 группах ({'; '.join(f'{g}: {n}' for g, n in feat_groups.items())}); {n_cards} вошли в скор-карту. Единый расчёт (`riskscore/features.py`) для обучения и инференса. Словарь — `feature-dictionary.md`. Пропуски: нет кредитной истории ≠ 0 (индикатор `has_credit_history` + отдельная WOE-корзина «нет данных»); выбросы клиппируются по квантилям 0.5/99.5% train.

## Алгоритм
1. Признаки → клиппинг по квантилям → **монотонный WOE-биннинг** (ожидаемый знак влияния задан словарём) → отбор: IV ≥ 0.02, |ρ| < 0.75, положительные коэффициенты;
2. **Логистическая регрессия** (L2, C = {tm['woe_logit_C']}) на WOE;
3. **Калибровка Платта** на valid → `pd` (границы {0.002:.1%}…95%);
4. **Скор-карта**: `riskScore = 600 − 50/ln2 · (logit(PD) − logit(5%))`, диапазон 300–900, больше = надёжнее; вклад каждого признака в баллах (`scorecard.md`);
5. **Рейтинг** — 10 градаций AAA…D по PD (`rating-scale.md`).
**Почему LR:** прозрачна для андеррайтера, напрямую переводится в скор-карту, устойчива и стабильна.

## Метрики (test, отложенная выборка)
Gini **{T['gini']:.3f}** [{ci['gini'][0]:.3f}; {ci['gini'][1]:.3f}] · ROC-AUC {T['auc']:.3f} · KS {T['ks']:.3f} · Brier {T['brier']:.4f} · ECE {T['ece']:.4f} · PSI(train→test) {psi['score_train_test']:.4f}. Рейтинги монотонны. Подробно — `model-report.md`.

## Ограничения и риски
- Синтетика: метрики не переносятся; закономерности заданы авторами.
- Малое число редких событий (блокировки, банкротства); верхние рейтинги содержат единицы дефолтов.
- Горизонт 12 мес., без макрофакторов; PD калибрована на среднем уровне ≈ 10% — в другой фазе цикла калибровку нужно пересчитать.
- Рекомендованные лимиты/ставки — формульные ориентиры (допущения о ставках и коэффициентах оборотов), а не банковский прайсинг.
- Риск «перенастройки»: LR восстанавливает собственную порождающую модель; на реальных данных возможна потеря качества.
- Справедливость: персональных признаков нет, но отрасль/регион могут нести косвенные эффекты — на реальных данных нужен fairness-анализ.

## Как развивать и мониторить в проде
- **Мониторинг:** PSI скора и признаков (ежемесячно, пороги 0.1/0.25), доля заявок с `warnings` (клиппинг/пропуски), доля пропусков БКИ, дрейф среднего PD vs факта; Gini/KS/Brier и таблица рейтингов на когортах созревших заявок (через 12 мес.); calibration-in-the-large.
- **Обновление:** переобучение по расписанию и по триггеру PSI > 0.25; версионирование моделей с откатом по `modelVersion`.
- **Развитие:** сегментные скор-карты ИП/ООО, подбор порогов под реальные потери (LGD, ставка), учёт обеспечения, временные ряды выручки, fairness-аудит, MLflow для трекинга.
- **Контроль версий:** артефакты `risk-scoring/models/*.joblib` + `scorecard_v1.json` + `rating_scale_v1.json`, `modelVersion` возвращается в каждом ответе.
"""
(ROOT / "docs/ml/model-card.md").write_text(CARD, encoding="utf-8")
print(len(CARD.splitlines()), "строк")

47 строк


## Итоговый отчёт `ОТЧЁТ_DS.md` (для сдачи/защиты)

In [6]:
from riskscore.engine import ScoringEngine
_eng = ScoringEngine(ROOT / "risk-scoring/models"); _te = ds[pd.read_csv(ROOT / "data/generated/split.csv").split.eq("test")]
_res = _eng.score_frame(_te, "2026-10-26T00:00:00Z"); zero_share = float(np.mean([r_["recommendedLimit"] == 0 for r_ in _res]))
cover_share = float(np.mean([r_["recommendedLimit"] >= q for r_, q in zip(_res, _te.requested_amount)]))
print(f"нулевой лимит у {zero_share:.1%} заявок test; лимит покрывает запрос у {cover_share:.1%}")
dec = demo.referenceDecision.value_counts().to_dict()
libs = "\n".join(f"| {p} | {vers.get(p, vers['nbclient'])} | {lic} |" for p, lic in [("numpy", "BSD-3"), ("pandas", "BSD-3"), ("scikit-learn", "BSD-3"), ("scipy", "BSD-3"), 
                                                                 ("matplotlib", "PSF-совместимая (Matplotlib License)"), ("joblib", "BSD-3"), ("pytest", "MIT"), ("nbclient (стек Jupyter)", "BSD-3")])
grades_t = md(pd.DataFrame({"Рейтинг": [g["grade"] for g in scale], "Сегмент": [g["segment"] for g in scale], "PD от": [f"{g['pd_from']:.1%}" for g in scale], "PD до": [f"{g['pd_to']:.1%}" for g in scale],
                            "Баллы": [f"{g['score_to']}–{g['score_from']}" for g in scale]}))
FINAL = f"""# Отчёт: Data Science / ML-часть проекта «Умный кредитный кабинет предпринимателя»

**Роль:** инженер скоринга и данных (Classic ML Engineer). **Хакатон:** Альфа-Банк × Changellenge, кейс «Цифровой кредитный конвейер для малого и микробизнеса».
**Дата:** {pd.Timestamp.today().strftime('%d.%m.%Y')} · **Модель:** `{m['model']}` · **Датасет:** `{m['dataset']}` (seed {m['seed']})

> ⚠️ **Все данные синтетические**, сгенерированы командой; модель обучена на синтетике, **метрики не переносятся на реальный портфель** — это демонстрация методологии.

---

## 1. Что сделано (кратко)
Полностью реализована DS-часть из ТЗ (`tz-ml-engineer.md`, §2.1–2.4 + движок оценки из §2.5) в виде 8 воспроизводимых Jupyter-ноутбуков:

| № | Ноутбук | Результат |
|---|---|---|
| 01 | `01_data_generation` | воспроизводимый генератор синтетики (`python -m data.generate --seed 42`, побайтово одинаков), {man['train_rows']} заявок + пул для демо, проверки качества |
| 02 | `02_features` | единый пайплайн признаков ({len(FEATURE_COLUMNS)} признаков, 5 групп), пропуски, клиппинг, split по компаниям, проверка утечек, словарь признаков |
| 03 | `03_modeling` | WOE-логрегрессия, калибровка, сериализация |
| 04 | `04_scorecard_rating` | балльная скор-карта (вклад каждого признака в баллах) и рейтинговая шкала из 10 градаций |
| 05 | `05_validation` | Gini, KS, Brier, ECE, calibration curve, PSI, сегменты, таблица рейтингов, permutation-важность, стресс-проверки |
| 06 | `06_scoring_engine` | движок: PD → рейтинг → лимит/ставка/срок → флаги политики → причины; что-если; {n_tests} автотестов (pytest) |
| 07 | `07_demo_cases_fixtures` | {len(demo)} демо-компаний (15 сценарных кейсов «ИНН → ожидаемое поведение»), JSON-фикстуры для моков и сидирования БД |
| 08 | `08_documentation` | model-report, model-card, data-generation, этот отчёт, Makefile, lock версий |

Принципиальная граница: скоринг отдаёт **оценку и рекомендацию** (`pd`, `riskScore`, `ratingGrade`, `recommendedLimit/Rate`, `policyFlags`, `reasons`); **кредитное решение** принимает `credit-core` — пороги одобрения в модель не зашиты.

## 2. Данные
**Откуда:** по условиям ТЗ используются только синтетические данные (покупные датасеты, реальные ИНН/выписки и доступ к системам банка запрещены), поэтому реальные данные не искались и не использовались. Данные сгенерированы по собственной порождающей модели, с открытыми справочниками (классификатор ОКВЭД, регионы РФ) только как каркасом.

**Что внутри** ({man['train_rows']} заявок, доля дефолтов {man['train_default_rate']:.1%}, заявки 2023-01…2025-09): профиль компании (ООО/ИП, ОКВЭД→11 отраслей, регион, численность, налоговый режим, статус); финансы за 8 кварталов (активы = обязательства + капитал выполняется точно); обороты по счёту за 12 мес.; кредитная история БКИ (просрочки 30/60/90+, запросы, дефолты; у ~{no_hist:.0%} истории нет); данные ФНС (налоговый долг, блокировки, массовый адрес/руководитель, дисквалификация, банкротство); параметры заявки (4 продукта, сумма, срок, обеспечение).

**Как порождён `default_flag`:** скрытое «качество бизнеса» + осмысленные факторы (нагрузка платежи/EBITDA, просрочки, волатильность и падение выручки, возраст бизнеса, налоговый долг, блокировки, ликвидность…) + шум → Bernoulli. Gini «оракула» (знает формулу наблюдаемой части риска) ≈ {oracle_gini:.2f}, так что модель не может быть идеальной. ИНН/ОГРН имеют валидные контрольные суммы и код региона `00` (несуществующий). Подробности — `docs/ml/data-generation.md`.

## 3. Модель и почему
Выбрана **WOE-логрегрессия** (монотонный биннинг, L2, отбор по IV и корреляциям): она интерпретируема и напрямую переводится в скор-карту (вклад каждого признака в баллах), что и требует ТЗ. Поверх неё — **калибровка Платта** → `pd` как настоящая вероятность. Сверху — **скор-карта** (600 баллов = PD 5%, +50 баллов = удвоение шансов) и **10 рейтингов** AAA…D.

Дополнительно реализован слой рекомендаций: лимит = min(лимит по денежному потоку (PV аннуитета от `share·EBITDA − текущие платежи`), лимит по оборотам, потолок продукта); ставка = база продукта + надбавка за риск; срок убывает с ростом PD; 16 флагов политики (CRITICAL/WARNING/INFO); причины решения (≥ 5 факторов с баллами и текстом).

## 4. Результаты метрик (отложенная тестовая выборка, {T['n']} заявок)
| Метрика | Значение | 95% ДИ | Комментарий |
|---|---|---|---|
| **Gini** | **{T['gini']:.3f}** | [{ci['gini'][0]:.3f}; {ci['gini'][1]:.3f}] | требование ТЗ ≥ 0.45 — выполнено |
| ROC-AUC | {T['auc']:.3f} | | |
| **KS** | **{T['ks']:.3f}** | [{ci['ks'][0]:.3f}; {ci['ks'][1]:.3f}] | |
| **Brier** | **{T['brier']:.4f}** | [{ci['brier'][0]:.4f}; {ci['brier'][1]:.4f}] | Brier Skill Score {m['brier_skill']:.3f} |
| ECE (калибровка) | {T['ece']:.4f} | | средний PD {T['mean_pd']:.2%} vs факт {m['default_rate']['test']:.2%} |
| **PSI** скора train→test | **{psi['score_train_test']:.4f}** | | < 0.1 — стабильно; максимум PSI признака {psi['features_max']:.4f} |
| Gini train / valid | {TR['gini']:.3f} / {V['gini']:.3f} | | разрыв train→test {gap:+.3f} |
| Монотонность рейтингов | **да** (test и вся выборка) | | факт. доля дефолтов растёт от AAA к D |
| Gini по сегментам | {gseg.min():.2f}–{gseg.max():.2f} | | {int((gseg >= 0.45).sum())} из {len(gseg)} сегментов ≥ 0.45 (отрасль, форма, размер, продукт, регион, год) |

Таблица рейтингов (test): монотонность подтверждена, средний PD внутри 95% ДИ факта {in_ci_txt}:

{md(rt_t)}

**Топ-факторы PD:** по скор-карте — {top_card}; по permutation importance на test — {top_perm}. Блоки совпадают по смыслу в двух способах оценки (долговая нагрузка и покрытие платежей, прибыльность/убыточность, автономия, возраст, просрочки); плюс стресс-тесты (просрочка 90+ → ΔPD {m['sensitivity']['просрочка 90+ (150 дн.)']:+.3f}, блокировка счёта → {m['sensitivity']['блокировка счёта']:+.3f}, падение выручки −30% → {m['sensitivity']['выручка последнего кв. −30%']:+.3f}) — все знаки верные.

Графики: `docs/ml/img/` (ROC/KS, calibration, рейтинги, важность). Полный отчёт: `docs/ml/model-report.md`.

![ROC и KS](docs/ml/img/05_roc_ks.png)
![Calibration](docs/ml/img/05_calibration.png)
![Рейтинги](docs/ml/img/05_rating_table.png)

### Рейтинговая шкала
{grades_t}

## 5. Демо-кейсы и фикстуры
{len(demo)} демо-компаний с едиными ИНН во всех файлах `data/fixtures/` (`companies.json` для сидирования БД; `egrul/financials/bki/fns/bank_turnover.json` для моков; `companies_flat.csv`; `demo_cases.csv`). 15 сценарных кейсов с заранее известным исходом: автоодобрение (2), контр-оферта по лимиту, серая зона → ручное рассмотрение, отказ по риску, нет кредитной истории, ликвидация, блокировка счёта, налоговый долг, просрочка 90+, массовый адрес, дисквалифицированный руководитель, молодой бизнес, банкротство, отрицательный капитал + 50 типичных (по 5 на рейтинг). Таблица «ИНН → ожидаемое поведение» — `docs/test-data.md`. Референсные решения ({', '.join(f'{k}: {v}' for k, v in dec.items())}) — иллюстрация политики `credit-core`.

## 6. Стек технологий
Python {platform.python_version()}, Jupyter-ноутбуки; **pandas/numpy** (данные), **scikit-learn** (Pipeline, LogisticRegression, метрики), собственные модули (`riskscore`: признаки, WOE-кодировщик с монотонным биннингом, калиброванная модель, движок), **matplotlib** (графики), **joblib** (сериализация), **pytest** ({n_tests} тестов). Только свободные библиотеки; платных API, внешних датасетов и LLM не использовалось.

| Библиотека | Версия | Лицензия |
|---|---|---|
{libs}

Версии зафиксированы в `risk-scoring/requirements-ds.lock.txt`. Лицензии — для `THIRD_PARTY_NOTICES` TeamLead'а.

## 7. Структура и воспроизведение
```
data/generate.py, fixtures.py   генератор и конвертер фикстур      data/fixtures/   JSON/CSV для моков и БД
data/generated/                  датасеты (CSV), manifest.json, split.csv
notebooks/01…08_*.ipynb          весь код проекта
risk-scoring/riskscore/          features.py, woe.py, models.py, metrics.py, engine.py
risk-scoring/models/             модели (.joblib), scorecard_v1.json, rating_scale_v1.json, training_meta.json
risk-scoring/tests/              {n_tests} pytest-тестов
docs/ml/                         model-report, model-card, feature-dictionary, rating-scale, scorecard, data-generation, img/
docs/test-data.md                ИНН → ожидаемое поведение
```
Воспроизведение: `make train` (данные → модель), `make report` (метрики, фикстуры, документация), `make test`. Ноутбуки запускались **по одной ячейке** (последовательно в одном ядре), потоки ограничены (2), объём данных мал — нагрузка на компьютер небольшая.

## 8. Что не вошло / ограничения (честно)
- **HTTP-сервис FastAPI, Dockerfile, `uv.lock`/CI** (§2.5.1–2.5.14 ТЗ) не реализованы: по заданию сделана DS-часть в ноутбуках. Вся логика ответа `/score` уже готова в `engine.py` (`score_one`, `score_frame`, `suggest_conditions`, `model_info`) — сервис это тонкая обёртка (pydantic-валидация, `X-Correlation-Id`, `/health`), которую можно добавить за часы. Время одиночного скоринга ≈ 75 мс (p95 < 300 мс), загрузка модели один раз.
- Дополнительные задания: реализовано «что-если» (`suggest_conditions`), объяснение по баллам, шкала рейтингов, PSI-мониторинг как метрика; не делались сегментные модели, shadow-эндпоинт, OCR, LLM-ассистент, MLflow.
- Данные и метрики синтетические; при замене на реальные данные нужны переобучение, независимая валидация и fairness-аудит.
- Около {zero_share:.0%} заявок получают нулевой рекомендованный лимит по формуле платёжной способности (текущие долги + новый платёж ≤ доля EBITDA), у {cover_share:.0%} лимит покрывает запрошенную сумму; коэффициенты формулы вынесены в конфигурацию `PRODUCT_CONFIG` и настраиваются.
- Модель восстанавливает нами же заданные закономерности; высокие метрики на синтетике — не ожидание на реальных данных.
"""
(ROOT / "ОТЧЁТ_DS.md").write_text(FINAL, encoding="utf-8")
print(len(FINAL.splitlines()), "строк; записано ОТЧЁТ_DS.md")

нулевой лимит у 40.3% заявок test; лимит покрывает запрос у 42.5%
125 строк; записано ОТЧЁТ_DS.md
